In [1]:
# Using Lucas-Kanade function

In [2]:
import cv2
import numpy as np

### Sparse Optical Flow in OpenCV

`calcOpticalFlowPyrLK(prevImg, nextImg, prevPts, nextPts, status, err[, winSize[, maxLevel[, criteria[, flags[, minEigThreshold]]]]]) -> nextPts, status, err`

This function computes a sparse optical flow using the Lucas-Kanade method.

Here are the parameters for the function and what they represent:

- **prevImg** first 8-bit input image (single-channel).  
- **nextImg** second input image of the same size and type as prevImg.  
- **prevPts** vector of 2D points for which the flow needs to be found.  
- **nextPts** output vector of 2D points containing the calculated new positions of input features in the next image.  
- **status** output status vector (of unsigned chars); each element of the vector is set to 1 if the flow for the corresponding features has been found, otherwise 0.  
- **err** output vector of errors; each element contains the difference between the original and moved points.  

Optional parameters:

- **winSize** size of the search window at each pyramid level. Larger values may improve robustness but increase computation.  
  - Typical value: (21, 21)  
- **maxLevel** 0-based maximum pyramid level number.  
  - If set to 0, no pyramids are used (i.e., single-level).  
- **criteria** termination criteria of the iterative search algorithm (after specified max count or epsilon is reached).  
  - Typical value: `criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)`  
- **flags** operation flags:
  - `cv2.OPTFLOW_USE_INITIAL_FLOW`: Use initial values in `nextPts` as initial estimates.  
  - `cv2.OPTFLOW_LK_GET_MIN_EIGENVALS`: Get minimum eigen values for each point.
- **minEigThreshold** minimal eigenvalue threshold that defines whether a point is considered for tracking.  
  - Typical value: 1e-4


In [3]:
corner_track_params = dict(maxCorners=10, qualityLevel=0.03, minDistance=7, blockSize=7)

In [4]:
lk_params = dict(winSize=(200,200), maxLevel=2, criteria = (cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,10,0.03))

In [5]:
cap = cv2.VideoCapture(0)

ret, prev_frame = cap.read()
prev_frame_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# pts to track
prevPts = cv2.goodFeaturesToTrack(prev_frame_gray, mask = None, **corner_track_params)

mask = np.zeros_like(prev_frame)

while True:
    ret, curr_frame = cap.read()
    curr_frame_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    
    # if flow found in corresponding feature, status will be 1, else 0
    nextPts, status, err = cv2.calcOpticalFlowPyrLK(prev_frame_gray, curr_frame_gray, prevPts,None, **lk_params) # nextPts = Nune here
    
    good_prev = prevPts[status == 1] 
    good_new = nextPts[status == 1]
    
    for i, (prev, new) in enumerate(zip(good_prev, good_new)):
        
        x_prev, y_prev = prev.ravel()
        x_new, y_new = new.ravel()
        
        mask = cv2.line(mask, (x_prev, y_prev), (x_new, y_new), (0,255,0), 2)
        
        curr_frame = cv2.circle(curr_frame, ( x_new, y_new), 8, (0,0,255), -1)
        
    img = cv2.add(curr_frame, mask)
    cv2.imshow('Tracking', img)
    
    k = cv2.waitKey(30) & 0xFF
    if k == 27:
        break
        
    prev_frame_gray = curr_frame_gray.copy()
    prevPts = good_new.reshape(-1,1,2)
        
cap.release()
cv2.destroyAllWindows()

### Dense Optical Flow in OpenCV

`calcOpticalFlowFarneback(prev, next, flow, pyr_scale, levels, winsize, iterations, poly_n, poly_sigma, flags) -> flow`

This function computes a dense optical flow using the Gunnar Farneback's algorithm.

Here are the parameters for the function and what they represent:

- **prev** first 8-bit single-channel input image.  
- **next** second input image of the same size and the same type as prev.  
- **flow** computed flow image that has the same size as prev and type CV_32FC2.  
- **pyr_scale** parameter, specifying the image scale (<1) to build pyramids for each image  
  - `pyr_scale=0.5` means a classical pyramid, where each next layer is twice smaller than the previous one.  
- **levels** number of pyramid layers including the initial image; levels=1 means that no extra layers are created and only the original images are used.  
- **winsize** averaging window size  
  - larger values increase the algorithm robustness to image  
  - noise and give more chances for fast motion detection, but yield more blurred motion field.  
- **iterations** number of iterations the algorithm does at each pyramid level.  
- **poly_n** size of the pixel neighborhood used to find polynomial expansion in each pixel  
  - larger values mean that the image will be approximated with smoother surfaces, yielding more robust algorithm and more blurred motion field, typically poly_n =5 or 7.  
- **poly_sigma** standard deviation of the Gaussian that is used to smooth derivatives used as a basis for the polynomial expansion; for poly_n=5, you can set poly_sigma=1.1, for poly_n=7, a good value would be poly_sigma=1.5.